# Practice: Transfer Learning with ResNet18

Use this practice after Section 7. It runs independently of the Section 6 practice.

- Download and extract the digit images.
- Prepare image batches using the pretrained weights' transforms.
- Replace the ImageNet head with a ten-class classifier and freeze the backbone.
- Train the replacement classification head.
- Measure accuracy on held-out digits.
- Save the trained model and reload it to classify one image.

Run the notebook from top to bottom in a fresh runtime. The first run also downloads about 45 MB of ResNet18 weights. A GPU is optional.

Use Colab's standard Python runtime. For local Jupyter, make sure `sample_data/` exists in the working directory, `curl` and `unzip` are available, and the required packages are installed (`python -m pip install torch torchvision scikit-learn matplotlib`).

[Open in Colab](https://colab.research.google.com/github/wlos/ai_with_python_course/blob/main/16-Deep-Learning-and-Image-Processing/practice/Practice%20Transfer%20Learning%20with%20ResNet18.ipynb)


## 1. Download and extract the digit images

The `!` prefix runs a terminal command from a notebook cell. The cell downloads `digit_images.zip` into Colab's `sample_data/` folder and extracts it to `sample_data/digit_images/`. The `-n` option skips files that already exist.


In [ ]:
!curl -fLsS -o sample_data/digit_images.zip https://raw.githubusercontent.com/wlos/AI_SS/main/13-Computer-Vision/sample_data/digit_images.zip
!unzip -q -n sample_data/digit_images.zip


## 2. Prepare inputs for ResNet18

Predict how many channels the pretrained model expects. Run the cell to check the image shape. Identify where the code selects the resize, crop and normalisation associated with the pretrained weights.


In [ ]:
from torchvision import datasets, models

DIGIT_ROOT = "sample_data/digit_images"
resnet_weights = models.ResNet18_Weights.DEFAULT
digit_dataset = datasets.ImageFolder(DIGIT_ROOT, transform=resnet_weights.transforms())
resnet_image, label = digit_dataset[0]
print(f"Input shape: {tuple(resnet_image.shape)} | Label: {label}")
assert len(digit_dataset) == 240 and len(digit_dataset.classes) == 10
assert tuple(resnet_image.shape) == (3, 224, 224)


## 3. Split and batch the images

Use `random_split` with the same 80/20 split sizes and seed as in Section 6. Record the batch shape and explain why the same image files produce different input tensors for ResNet18 and for the simple CNN.


In [ ]:
import torch
from torch.utils.data import DataLoader, random_split

DIGIT_SEED = 42

# Randomly assign each image to exactly one subset using a fixed seed.
train_size = int(0.8 * len(digit_dataset))
test_size = len(digit_dataset) - train_size
torch.manual_seed(DIGIT_SEED)
digit_train, digit_test = random_split(digit_dataset, [train_size, test_size])
digit_train_loader = DataLoader(digit_train, batch_size=16, shuffle=True)
digit_test_loader = DataLoader(digit_test, batch_size=16, shuffle=False)

batch_images, batch_labels = next(iter(digit_test_loader))
print(f"Training images: {len(digit_train)} | Test images: {len(digit_test)}")
print(f"Image batch: {tuple(batch_images.shape)} | Label batch: {tuple(batch_labels.shape)}")
print(f"Batch labels: {batch_labels.tolist()}")


## 4. Set up the trainable head

Identify which line freezes the existing parameters and which line selects the parameters for Adam. Explain how `eval()` protects BatchNorm running statistics while the optimiser updates the new head.


In [ ]:
from torch import nn

DIGIT_LEARNING_RATE = 0.001
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
digit_class_count = len(digit_dataset.classes)
torch.manual_seed(DIGIT_SEED)
digit_resnet = models.resnet18(weights=resnet_weights)
digit_resnet.requires_grad_(False)  # Freeze all original pretrained parameters.
digit_resnet.fc = nn.Linear(digit_resnet.fc.in_features, digit_class_count)
digit_resnet = digit_resnet.to(DEVICE)
digit_resnet.eval()  # Preserve the backbone's BatchNorm running statistics.
digit_resnet.fc.train()  # Set training mode only for the replacement head.
resnet_optimiser = torch.optim.Adam(digit_resnet.fc.parameters(), lr=DIGIT_LEARNING_RATE)
print(f"Trainable head parameters: {sum(p.numel() for p in digit_resnet.fc.parameters()):,}")


## 5. Train the classification head

Train the classification head on the training batches and record the mean loss after each epoch.


In [ ]:
DIGIT_EPOCHS = 3
digit_loss = nn.CrossEntropyLoss()

def train_digit_epoch(model, loader, optimiser):
    '''Train for one epoch and return mean loss and accuracy.'''
    total_loss = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimiser.zero_grad()  # Clear gradients left by the previous batch.
        logits = model(images)
        loss = digit_loss(logits, labels)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        loss.backward()  # Calculate gradients from the loss.
        optimiser.step()  # Apply an update to the optimiser's parameters.
        total_loss += loss.item() * len(labels)  # Weight each batch by its size.
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

# Restore the intended modes if you rerun this cell after evaluating.
digit_resnet.eval()
digit_resnet.fc.train()
resnet_training_losses = []
resnet_training_accuracies = []
for epoch in range(DIGIT_EPOCHS):
    mean_loss, train_accuracy = train_digit_epoch(digit_resnet, digit_train_loader, resnet_optimiser)
    resnet_training_losses.append(mean_loss)
    resnet_training_accuracies.append(train_accuracy)
    print(f"Head epoch {epoch + 1}: loss={mean_loss:.3f} | accuracy={train_accuracy:.1%}")


The curves track mean loss and accuracy on the training batches. Accuracy is the fraction of correct predictions made before each batch's weight update. The model is evaluated on held-out images in the next step.


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(resnet_training_losses) + 1)
figure, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(epochs, resnet_training_losses, marker="o")
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Mean cross-entropy loss", xticks=list(epochs))
axes[1].plot(epochs, [100 * accuracy for accuracy in resnet_training_accuracies], marker="o")
axes[1].set(title="Training accuracy", xlabel="Epoch", ylabel="Accuracy (%)", xticks=list(epochs), ylim=(0, 100))

figure.tight_layout()
plt.show()


## 6. Test and evaluate on held-out digits

Run ResNet18 on all 48 test images, then use scikit-learn to calculate overall test accuracy. Compare the results with the simple CNN if you completed the Section 6 practice. Name one reason ImageNet features might help and one reason a small test set makes the comparison uncertain.


In [ ]:
def predict_digits(model, loader):
    """Return true and predicted class indices in test-loader order."""
    model.eval()
    true_labels = []
    predictions = []
    with torch.no_grad():  # Inference does not need gradients.
        for images, labels in loader:
            logits = model(images.to(DEVICE))
            true_labels.extend(labels.tolist())
            predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return true_labels, predictions

resnet_test_labels, resnet_test_predictions = predict_digits(digit_resnet, digit_test_loader)
print(f"Test predictions: {len(resnet_test_predictions)}")


In [ ]:
from sklearn.metrics import accuracy_score

resnet_digit_accuracy = accuracy_score(resnet_test_labels, resnet_test_predictions)
print(f"Frozen ResNet18 test accuracy: {resnet_digit_accuracy:.1%}")


The grid shows the first eight test images in test-loader order, with the true and predicted digit above each image. Compare the predictions with the original images.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

figure, axes = plt.subplots(2, 4, figsize=(10, 5))
for index, axis in enumerate(axes.flat):
    # Look up the original file using the same order as the test predictions.
    image_path, _ = digit_dataset.samples[digit_test.indices[index]]
    with Image.open(image_path) as image:
        axis.imshow(image.convert("L"), cmap="gray")
    axis.set_title(f"True: {resnet_test_labels[index]} | Predicted: {resnet_test_predictions[index]}")
    axis.axis("off")

figure.tight_layout()
plt.show()


## 7. Save and reuse the trained model

Save the model's `state_dict`, which contains its trained parameters and buffers, including BatchNorm running statistics. Also save the class names in output order and the name of the pretrained weights so the same image transform can be recovered later.

The file `resnet18_digits.pth` is written to the current working directory. In Colab, download it from the **Files** panel to keep it after the runtime ends.


In [ ]:
MODEL_FILE = "resnet18_digits.pth"

torch.save({
    "model_state_dict": digit_resnet.state_dict(),
    "class_names": digit_dataset.classes,
    "preprocessing_weights": resnet_weights.name,
}, MODEL_FILE)
print(f"Saved model to {MODEL_FILE}")


To classify an image in a fresh session, place the saved file and the example image at the paths below. Recreate the ResNet18 architecture with the same number of outputs, then load the saved state onto the CPU. `weights=None` creates the architecture without downloading ImageNet weights. Apply the saved preprocessing and add a batch dimension before making the prediction.

[PyTorch guide to saving and loading models](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)


In [ ]:
import torch
from PIL import Image
from torch import nn
from torchvision import models

MODEL_FILE = "resnet18_digits.pth"
IMAGE_FILE = "sample_data/digit_images/0/mnist-row-00106.png"

checkpoint = torch.load(MODEL_FILE, map_location="cpu", weights_only=True)
class_names = checkpoint["class_names"]
restored_resnet = models.resnet18(weights=None)
restored_resnet.fc = nn.Linear(restored_resnet.fc.in_features, len(class_names))
restored_resnet.load_state_dict(checkpoint["model_state_dict"])
restored_resnet.eval()  # Use the saved BatchNorm statistics.

weights = models.ResNet18_Weights[checkpoint["preprocessing_weights"]]
with Image.open(IMAGE_FILE) as image:
    image_batch = weights.transforms()(image.convert("RGB")).unsqueeze(0)

with torch.no_grad():
    predicted_index = restored_resnet(image_batch).argmax(dim=1).item()
print(f"Predicted digit: {class_names[predicted_index]}")
